In [14]:
import pandas as pd
import sqlite3

In [15]:
#load

processor = pd.read_csv("/Users/yogesh/Downloads/processor_transactions.csv")
ledger = pd.read_csv("/Users/yogesh/Downloads/internal_ledger.csv")


# Normalize column names. Convert the column names to lower case

processor.columns = processor.columns.str.strip().str.lower()
ledger.columns = ledger.columns.str.strip().str.lower()

# Normalize status labels. processor says success and ledger says setttled
status_map = {"success": "settled", "failed": "failed", "refunded": "refunded"}
processor["status_normalized"] = processor["status"].map(status_map)

#Add a source tag
processor["source"] = "processor"
ledger["source"] = "ledger"
ledger["status_normalized"] = ledger["status"]

processor



,transaction_id,merchant,amount,currency,status,created_at,status_normalized,source
0,txn_0023,RetailPlus,38742,SGD,success,2024-01-10 17:00:00,settled,processor
1,txn_0064,CloudSoft,13969,EUR,success,2024-01-20 21:00:00,settled,processor
2,txn_0005,TravelHub,2239,INR,success,2024-01-29 02:00:00,settled,processor
3,txn_0065,Acme Corp,11360,USD,success,2024-01-20 22:00:00,settled,processor
4,txn_0075,HealthCo,7671,GBP,success,2024-01-12 02:00:00,settled,processor
...,...,...,...,...,...,...,...,...
100,txn_0091,RetailPlus,28193,INR,success,2024-01-02 13:00:00,settled,processor
101,txn_0063,TravelHub,7693,GBP,success,2024-01-12 16:00:00,settled,processor
102,txn_0067,CloudSoft,18617,EUR,success,2024-01-07 00:00:00,settled,processor
103,txn_0110,FoodMart,7047,EUR,failed,2024-01-29 08:00:00,failed,processor


In [16]:
# connect to DB

connection = sqlite3.connect("payments_recon.db")
processor.to_sql("processor_transactions", connection, if_exists="replace", index=False)
ledger.to_sql("internal_ledger", connection, if_exists="replace", index=False)

print("Data loaded Successfully")

Data loaded Successfully


In [22]:
# Reconcile steps

# Query to fetch missing entries from ledger
q1 = """
    SELECT 
	p.transaction_id,
	p.merchant,
	p.amount,
	p.status,
	p.created_at,
	p.status_normalized, "Missing from Ledger" AS mismatch_type
	FROM processor_transactions p
	LEFT JOIN internal_ledger l
	ON p.transaction_id = l.transaction_id
	WHERE l.transaction_id IS NULL
    ORDER BY p.created_at
"""

missing_from_ledger = pd.read_sql_query(q1, connection)

# print(f"Missing from ledger: {len(missing_from_ledger)} rows")
missing_from_ledger


,transaction_id,merchant,amount,status,created_at,status_normalized,mismatch_type
0,txn_0066,TechStore,21354,success,2024-01-04 12:00:00,settled,Missing from Ledger
1,txn_0072,CloudSoft,4394,success,2024-01-06 07:00:00,settled,Missing from Ledger
2,txn_0067,CloudSoft,18617,success,2024-01-07 00:00:00,settled,Missing from Ledger
3,txn_0099,FoodMart,11923,refunded,2024-01-09 13:00:00,refunded,Missing from Ledger
4,txn_0075,HealthCo,7671,success,2024-01-12 02:00:00,settled,Missing from Ledger
5,txn_0062,RetailPlus,3686,success,2024-01-12 12:00:00,settled,Missing from Ledger
6,txn_0063,TravelHub,7693,success,2024-01-12 16:00:00,settled,Missing from Ledger
7,txn_0073,FoodMart,4507,success,2024-01-12 23:00:00,settled,Missing from Ledger
8,txn_0105,TechStore,8252,refunded,2024-01-13 11:00:00,refunded,Missing from Ledger
9,txn_0102,CloudSoft,14159,refunded,2024-01-13 14:00:00,refunded,Missing from Ledger


In [24]:
# query to fetch missing data from Processor
q2 = """
    SELECT 
    p.transaction_id,
	l.transaction_id,
	l.merchant,
	l.amount,
	l.status,
	l.recorded_at,
	l.status_normalized, "Missing from Processor" AS mismatch_type
	FROM internal_ledger l
	LEFT JOIN processor_transactions p
	ON l.transaction_id = p.transaction_id 
	WHERE p.transaction_id IS NULL
"""

missing_from_processor = pd.read_sql_query(q2, connection)
missing_from_processor

,transaction_id,transaction_id,merchant,amount,status,recorded_at,status_normalized,mismatch_type
0,None,txn_0085,FoodMart,13816,settled,2024-01-11 11:00:00,settled,Missing from Processor
1,None,txn_0081,FoodMart,19587,settled,2024-01-19 17:00:00,settled,Missing from Processor
2,None,txn_0082,CloudSoft,4676,settled,2024-01-16 22:00:00,settled,Missing from Processor
3,None,txn_0078,HealthCo,9083,settled,2024-01-24 21:00:00,settled,Missing from Processor
4,None,txn_0079,Acme Corp,14514,settled,2024-01-06 03:00:00,settled,Missing from Processor
5,None,txn_0080,Acme Corp,2965,settled,2024-01-17 03:00:00,settled,Missing from Processor
6,None,txn_0077,RetailPlus,1944,settled,2024-01-06 22:00:00,settled,Missing from Processor
7,None,txn_0084,RetailPlus,8676,settled,2024-01-25 06:00:00,settled,Missing from Processor
8,None,txn_0083,TravelHub,1807,settled,2024-01-30 11:00:00,settled,Missing from Processor
9,None,txn_0076,MediaStream,2164,settled,2024-01-13 04:00:00,settled,Missing from Processor


In [29]:
# query to fetch mismatch in amount between processor and ledger

q3 = """
    SELECT 
    p.transaction_id,
    p.merchant,
    p.amount,
    p.status,
    p.amount - l.amount AS discrepency,
    "Amount Mismatch" AS mismatch_type
    FROM processor_transactions p
    INNER JOIN internal_ledger l
    ON p.transaction_id = l.transaction_id
    WHERE p.amount <> l.amount
"""
amount_mismatch = pd.read_sql_query(q3, connection)
amount_mismatch

,transaction_id,merchant,amount,status,discrepency,mismatch_type
0,txn_0094,TravelHub,9578,success,-500,Amount Mismatch
1,txn_0088,TravelHub,4537,success,-100,Amount Mismatch
2,txn_0090,TechStore,23512,success,-1000,Amount Mismatch
3,txn_0087,CloudSoft,28982,success,-500,Amount Mismatch
4,txn_0092,CloudSoft,13702,success,-1000,Amount Mismatch
5,txn_0089,TravelHub,22000,success,-500,Amount Mismatch
6,txn_0093,RetailPlus,30479,success,-1000,Amount Mismatch
7,txn_0095,TravelHub,2110,success,500,Amount Mismatch
8,txn_0086,MediaStream,12649,success,-100,Amount Mismatch
9,txn_0091,RetailPlus,28193,success,-100,Amount Mismatch


In [30]:

# q4 = """
#     SELECT
#     COUNT(*) FILTER (WHERE source = 'processor') AS processor_total,
#     COUNT(*) FILTER (WHERE source = 'ledger') AS ledger_total,
#     SUM(amount) FILTER (WHERE source = 'processor' AND status_normalized = 'settled') AS processor_settled_amount,
#     SUM(amount) FILTER (WHERE source = 'ledger' AND status_normalized = 'settled') AS ledger_settled_amount
#     FROM (
#     SELECT amount, status_normalized, 'processor' AS source FROM processor_transactions
#     UNION ALL
#     SELECT amount, status_normalized, 'ledger' AS source FROM internal_ledger
#     );
# """

# status_query = pd.read_sql_query(q4,connection)
# status_query

,processor_total,ledger_total,processor_settled_amount,ledger_settled_amount
0,105,90,1746009,1631790


In [8]:
# Summary Stats
q4 = """
    SELECT 
    COUNT(p.transaction_id) AS processor_total,
    COUNT(l.transaction_id) AS ledget_total,
    SUM(
        CASE 
            WHEN p.status_normalized = 'settled' THEN p.amount
            ELSE 0
        END
        ) AS processor_settled_amount,
    SUM(
        CASE
            WHEN l.status = 'settled' THEN l.amount
            ELSE 0
        END
        ) AS ledger_settled_amount
    FROM processor_transactions p
    FULL OUTER JOIN internal_ledger l
    ON p.transaction_id = l.transaction_id
"""

status_query = pd.read_sql_query(q4, connection)
status_query
    

,processor_total,ledget_total,processor_settled_amount,ledger_settled_amount
0,105,90,1746009,1631790


In [19]:
# Output a Mismatch Report

queries = {
    "missing_from_ledger" : """SELECT 
	p.transaction_id,
	p.merchant,
	p.amount,
	p.status,
	p.created_at,
	p.status_normalized, 'Missing from Ledger' AS mismatch_type
	FROM processor_transactions p
	LEFT JOIN internal_ledger l
	ON p.transaction_id = l.transaction_id
	WHERE l.transaction_id IS NULL
    ORDER BY p.created_at""",
    
    "missing_from_processor" : """SELECT 
	l.transaction_id,
	l.merchant,
	l.amount,
	l.status,
	l.recorded_at,
	l.status_normalized, "Missing from Processor" AS mismatch_type
	FROM internal_ledger l
	LEFT JOIN processor_transactions p
	ON l.transaction_id = p.transaction_id 
	WHERE p.transaction_id IS NULL""",

    "amount_mismatch" : """SELECT 
    p.transaction_id,
    p.merchant,
    p.amount,
    p.status,
    p.amount - l.amount AS discrepency,
    "Amount Mismatch" AS mismatch_type
    FROM processor_transactions p
    INNER JOIN internal_ledger l
    ON p.transaction_id = l.transaction_id
    WHERE p.amount <> l.amount"""
}


output_file = "Recon_Report.xlsx"

with pd.ExcelWriter(output_file, engine = "xlsxwriter") as writer:
    for sheet_name, query in queries.items():
        print(f"Running: {sheet_name}")
        df = pd.read_sql_query(query, connection)
        df.to_excel(writer, sheet_name = sheet_name, index = False)

print("Reconciliation Report generated.")
        




Running: missing_from_ledger
Running: missing_from_processor
Running: amount_mismatch
Reconciliation Report generated.


In [20]:
import os
print(os.getcwd())

/Users/yogesh
